# GPU-accelerated conveyor waste pipeline

Real-time waste detection on a conveyor belt, with each CUDA optimisation measured
rather than asserted. **Prototype, not a deployment.**

**Before running:** `Runtime > Change runtime type > Hardware accelerator = T4 GPU`.

This notebook is the single entry point. It compiles the Phase A kernels with `nvcc`,
builds the fused preprocess kernel as a PyTorch extension, runs the inference ladder,
the pipelined loop and the batching experiment, then generates `results/summary.md`
from the CSVs that were just written.

Nothing in the write-up is typed by hand: every table is generated from
`results/*.csv`, and a benchmark that fails is recorded as a failure in
`results/failures.md` instead of being quietly dropped.

| Stage | What it proves | ~time |
|---|---|---|
| 1. Environment | T4, CUDA toolkit, Nsight availability | 1 min |
| 2. Phase A | vector add, 3 reductions, 3 matmuls, streams | 3 min (incl. compile) |
| 3. Fused preprocess | one kernel vs OpenCV/NumPy on CPU | 3 min |
| 4. Inference ladder | FP32 -> FP16 -> ONNX RT -> TensorRT | 6 min |
| 5. Pipeline + batching | serial vs 2/3 streams, 1/2/4 feeds | 4 min |
| 6. Profiling | Nsight Systems; Nsight Compute is attempted and reported honestly | 3 min |
| 7. Summary | `results/summary.md` from the CSVs | 5 s |


In [9]:
# ---- 1. Bootstrap: find the repo, get onto it, expose phase_b on sys.path ----
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()

# Optional: set this to your repo URL and the notebook clones it when the files
# are not already present (e.g. a fresh Colab runtime).
GIT_URL = "https://github.com/pkbros/CUDA-accelerated-waste-detection-pipeline.git"


def find_repo_root(start=None):
    """Walk up from `start` until the repo marker file is found."""
    here = Path(start or Path.cwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "include" / "bench_common.h").exists():
            return cand
    return None


REPO = find_repo_root()

if REPO is None and IN_COLAB:
    # A folder uploaded through the Files sidebar lands under /content/<name>/.
    for cand in Path("/content").glob("*/include/bench_common.h"):
        REPO = cand.parent.parent
        break

if REPO is None and IN_COLAB:
    target = Path("/content/cuda-waste-pipeline")
    if not GIT_URL:
        raise SystemExit(
            "Repo not found. Either upload the project folder through the Files sidebar, "
            "or set GIT_URL at the top of this cell and re-run."
        )
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(target)], check=True)
    REPO = find_repo_root(target)

if REPO is None:
    raise SystemExit("Run this notebook from inside the repo (include/bench_common.h missing).")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "phase_b"))
(REPO / "results").mkdir(exist_ok=True)
(REPO / "build").mkdir(exist_ok=True)
(REPO / "reports").mkdir(exist_ok=True)


class CmdResult:
    """Minimal stand-in for CompletedProcess; keeps .stdout for callers."""
    def __init__(self, returncode, stdout):
        self.returncode = returncode
        self.stdout = stdout


def sh(cmd, cwd=None, check=True, quiet=False):
    """Run a shell command, echo it, STREAM its output, raise on failure.

    subprocess.run() with stdout=PIPE buffers everything until the child exits,
    which makes a four-minute benchmark look hung and tempts people to
    interrupt it. Popen + line-by-line printing shows progress immediately.
    PYTHONUNBUFFERED matters too: Python children otherwise block-buffer their
    own stdout when it is a pipe.
    """
    print(f"$ {cmd}", flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd, shell=True, cwd=str(cwd or REPO), text=True, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    chunks = []
    for line in proc.stdout:
        chunks.append(line)
        if not quiet:
            print(line, end="", flush=True)
    proc.wait()
    result = CmdResult(proc.returncode, "".join(chunks))
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed ({result.returncode}): {cmd}")
    return result


def record_failure(stage, detail):
    """Failures are evidence. Write them down instead of hiding them."""
    path = REPO / "results" / "failures.md"
    new = not path.exists()
    with path.open("a", encoding="utf-8") as fh:
        if new:
            fh.write("# Recorded failures\n\nThings that did not work in this environment.\n")
            fh.write("Kept deliberately: the PRD asks for honesty over a clean story.\n\n")
        fh.write(f"- **{stage}**: {detail.strip()[:400]}\n")
    print(f"[recorded failure] {stage}: {detail.strip()[:200]}")


print("repo:", REPO)
print("colab:", IN_COLAB)


repo: /content/cuda-waste-pipeline
colab: True


## 1. Environment

What we need and why: `nvcc` to compile Phase A by hand, a PyTorch that agrees with the
system CUDA (otherwise `torch.utils.cpp_extension` refuses to build), and `nsys` for the
timeline. `ncu` (Nsight Compute) is attempted but is expected to fail on Colab because
performance counters need privileges a Colab runtime does not grant.


In [10]:
# ---- 1a. Device + toolchain report ----
import torch

print("torch      :", torch.__version__)
print("torch cuda :", torch.version.cuda)
print("cuda avail :", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then reconnect.")

props = torch.cuda.get_device_properties(0)
ARCH = f"sm_{props.major}{props.minor}"
print(f"gpu        : {props.name}  ({ARCH}, {props.multi_processor_count} SMs, "
      f"{props.total_memory / 1024**3:.1f} GiB)")

from common import write_device_info  # noqa: E402  (path set up in cell 1)

print("recorded   :", write_device_info())
sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")


torch      : 2.11.0+cu130
torch cuda : 13.0
cuda avail : True
gpu        : Tesla T4  (sm_75, 40 SMs, 14.6 GiB)
recorded   : {'gpu_name': 'Tesla T4', 'compute_capability': '7.5', 'arch': 'sm_75', 'multiprocessors': 40, 'global_mem_gib': 14.56, 'torch': '2.11.0+cu130', 'cuda_runtime': '13.0'}
$ nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB



CompletedProcess(args='nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv', returncode=0, stdout='name, driver_version, memory.total [MiB]\nTesla T4, 580.82.07, 15360 MiB\n')

In [11]:
# ---- 1b. System tools: nvcc (required), nsys / ncu (best effort) ----
def have(tool):
    return shutil.which(tool) is not None


print("nvcc :", shutil.which("nvcc") or "MISSING")
print("nsys :", shutil.which("nsys") or "MISSING")
print("ncu  :", shutil.which("ncu") or "MISSING")

# If nvcc is missing, install a toolkit whose MAJOR version matches torch, because
# torch.utils.cpp_extension raises when the extension's CUDA major differs from the
# one torch was built with. Installing CUDA 13 next to a cu12 torch is the classic
# way to break the preprocess build.
if not have("nvcc"):
    torch_cuda = torch.version.cuda or "12.4"
    major, minor = torch_cuda.split(".")[:2]
    print(f"installing a CUDA {major}.x compiler to match torch {torch_cuda}")
    sh("sudo apt-get update -qq", check=False)
    # Exact minor first; fall back to the major meta-package, because a given
    # minor version is not guaranteed to be in the apt repo. The MAJOR version is
    # what torch's extension build actually insists on matching.
    rc = sh(
        f"sudo apt-get install -y -qq cuda-compiler-{major}-{minor} cuda-cudart-dev-{major}-{minor}",
        check=False,
    ).returncode
    if rc != 0:
        rc = sh(f"sudo apt-get install -y -qq cuda-toolkit-{major}", check=False).returncode
    if rc != 0:
        record_failure("cuda toolkit install", f"apt could not install a CUDA {major}.x toolkit")
    # Optional extras: cuBLAS headers for matmul, CUB for the reduction baseline.
    if sh(f"sudo apt-get install -y -qq libcublas-dev-{major}-{minor} libcub-dev", check=False).returncode != 0:
        record_failure(
            "cuda extras",
            "libcublas-dev / libcub-dev not installed; matmul and the CUB reduction "
            "row may not build. The other kernels are unaffected.",
        )
    for candidate in sorted(Path("/usr/local").glob(f"cuda-{major}*")):
        if (candidate / "bin" / "nvcc").exists():
            os.environ["CUDA_HOME"] = str(candidate)
            os.environ["PATH"] = f"{candidate / 'bin'}{os.pathsep}{os.environ['PATH']}"
            break
    print("nvcc :", shutil.which("nvcc") or "STILL MISSING")

if not have("nvcc"):
    raise SystemExit("No nvcc: Phase A cannot be compiled. Fix the toolkit install above.")

os.environ.setdefault("CUDA_HOME", str(Path(shutil.which("nvcc")).resolve().parent.parent))
os.environ["CUDACXX"] = shutil.which("nvcc")
sh("nvcc --version | tail -n 2")
sh("make --version | head -n 1")

# Nsight Systems: present in the Colab image's CUDA apt repo. Non-fatal if absent.
if not have("nsys"):
    if sh("sudo apt-get install -y -qq nsight-systems", check=False).returncode != 0:
        sh("sudo apt-get install -y -qq nsight-systems-2026.1.3", check=False)
if have("nsys"):
    sh("nsys --version")

# Nsight Compute: install it, but expect ERR_NVGPUCTRPERM when it profiles, because
# GPU performance counters require privileges Colab does not give us. We try once here
# so the limitation is recorded with a real error message rather than an assumption.
if not have("ncu"):
    sh("sudo apt-get install -y -qq nsight-compute", check=False)
print("ncu  :", shutil.which("ncu") or "MISSING")


nvcc : /usr/local/cuda/bin/nvcc
nsys : /usr/local/bin/nsys
ncu  : /usr/local/cuda/bin/ncu
$ nvcc --version | tail -n 2
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0

$ make --version | head -n 1
GNU Make 4.3

$ nsys --version
NVIDIA Nsight Systems version 2025.3.1.0

ncu  : /usr/local/cuda/bin/ncu


## 2. Phase A — CUDA fundamentals

Four standalone `nvcc` builds, each with a CPU baseline and a correctness check against it.
Every binary appends rows to `results/*.csv` using the shared timing helpers in
`include/bench_common.h` (warmup, then one CUDA-event pair per iteration, mean ± sample stddev).

* **vector add** — sanity check, and the bandwidth ceiling that everything else is judged against.
* **reduction** — global memory → shared memory → warp shuffle. The story is how many global
  round trips and `__syncthreads()` calls each version removes.
* **matmul** — naive → 32×32 tiled shared memory → cuBLAS. cuBLAS is the honest ceiling;
  the remaining gap is register blocking, which this repo deliberately does not implement.
* **streams** — pageable vs pinned, 1 vs 2 vs 3 streams, H2D copy overlapping kernels.


In [12]:
# ---- 2a. Build Phase A ----
sh(f"make -C {shlex.quote(str(REPO / 'phase_a'))} ARCH={ARCH} -j2")


$ make -C /content/cuda-waste-pipeline/phase_a ARCH=sm_75 -j2
make: Entering directory '/content/cuda-waste-pipeline/phase_a'
make: Nothing to be done for 'all'.
make: Leaving directory '/content/cuda-waste-pipeline/phase_a'



CompletedProcess(args='make -C /content/cuda-waste-pipeline/phase_a ARCH=sm_75 -j2', returncode=0, stdout="make: Entering directory '/content/cuda-waste-pipeline/phase_a'\nmake: Nothing to be done for 'all'.\nmake: Leaving directory '/content/cuda-waste-pipeline/phase_a'\n")

In [13]:
# ---- 2b. Run Phase A, writing straight into results/ ----
phase_a = [
    ("vector_add", ["--n", "50000000", "--iters", "50"]),
    ("reduction",  ["--n", "16777216", "--iters", "50"]),
    ("matmul",     ["--n", "1024", "--cpu-n", "256", "--iters", "20"]),
    ("streams",    ["--total-mb", "64", "--iters", "20"]),
]

for name, args in phase_a:
    print(f"\n===== {name} =====")
    # The Makefile writes binaries to build/, not phase_a/.
    cmd = (
        f"{shlex.quote(str(REPO / 'build' / name))} "
        + " ".join(shlex.quote(a) for a in args)
        + f" --csv {shlex.quote(str(REPO / 'results' / (name + '.csv')))}"
    )
    # Run from phase_a/ so any relative path inside the binary still resolves.
    rc = sh(cmd, cwd=REPO / "phase_a", check=False).returncode
    if rc != 0:
        record_failure(f"phase A / {name}", f"exit code {rc}")
        print(f"{name}: FAILED (recorded)")



===== vector_add =====
$ /content/cuda-waste-pipeline/build/vector_add --n 50000000 --iters 50 --csv /content/cuda-waste-pipeline/results/vector_add.csv
vector_add: N=50000000 floats (190.7 MiB/array), iters=50, warmup=10, block=256, rtol=1e-05
device: Tesla T4 (sm_75)
  cpu_reference             0.0000 ms  +-0.0000    (n=0)       0.00 GB/s
  [verify] gpu_scalar    max_abs_err=0, rel_err=0, ok=true
  gpu_scalar                2.2859 ms  +-0.0025    (n=50)     262.47 GB/s
  [verify] gpu_float4    max_abs_err=0, rel_err=0, ok=true
  gpu_float4                2.3892 ms  +-0.0052    (n=50)     251.13 GB/s
wrote 3 measured rows to /content/cuda-waste-pipeline/results/vector_add.csv


===== reduction =====
$ /content/cuda-waste-pipeline/build/reduction --n 16777216 --iters 50 --csv /content/cuda-waste-pipeline/results/reduction.csv
reduction: N=16777216 threads=256 iters=50 warmup=10 tol=0.0001 csv=/content/cuda-waste-pipeline/results/reduction.csv
  analytic=24117248.000000  cpu_double=241

## 3. Fused preprocess kernel

One kernel doing five things that normally cost five passes over memory: bilinear letterbox
resize, BGR→RGB, normalise, HWC→CHW, uint8→FP16/FP32.

Correctness is checked against an independent CPU implementation with a **stated tolerance** —
`atol=rtol=1e-4` for FP32, `atol=2e-3` for FP16 — before any timing is recorded. A fast kernel
that produces the wrong tensor is not a result.


In [14]:
!pip -q install ninja

In [15]:
# ---- 3. Build the extension (first build ~60-90 s; cached afterwards) ----
# torch.utils.cpp_extension uses ninja as its build backend, and Colab's base image
# does not ship it. Without this the build dies before nvcc is ever invoked.
if shutil.which("ninja") is None:
    sh("pip -q install ninja", check=False)
print("ninja :", shutil.which("ninja") or "STILL MISSING")

# A failure here is recorded and the notebook continues: phase_b/pipeline.py falls
# back to a torch preprocess and labels that fact in its CSV rows, so the rest of
# the run still produces honest numbers.
try:
    import preprocess_ext as pp
    pp.main()
except Exception as exc:  # noqa: BLE001
    record_failure("fused preprocess kernel", repr(exc))
    print("preprocess kernel unavailable; pipeline will use the torch fallback and say so.")


ninja : /usr/local/bin/ninja
Device: Tesla T4  arch=sm_75  torch=2.11.0+cu130
[recorded failure] fused preprocess kernel: RuntimeError('Could not build phase_b/preprocess.cu. No fallback was used. The PRD fallback is a CuPy RawKernel; implementing it is a separate, explicitly-reported task, not a silent substitution.')
preprocess kernel unavailable; pipeline will use the torch fallback and say so.


CUDA EXTENSION COMPILE FAILED -- full error below:
/content/cuda-waste-pipeline/.cache/torch_extensions/waste_preprocess.so: cannot open shared object file: No such file or directory


## 4. Inference ladder

YOLOv8n on the same 640×640 input, four backends. The PRD treats TensorRT as a stretch goal
because installing it on Colab can pull in CUDA packages that fight the preinstalled PyTorch,
and ONNX Runtime's CUDA provider sometimes needs its library path fixed. Both are attempted;
if they fail the reason is written to `results/failures.md` and no row is emitted for them.

**No row means no claim.** That is the whole point of the rule.


In [16]:
# ---- 4a. Python deps for the ladder ----
# Record the torch version before and after: if pip upgrades torch, every result
# measured earlier in this session came from a different build and the comparison
# would be invalid. Cheap insurance against a very confusing table.
torch_before = torch.__version__
sh("pip -q install ultralytics onnx onnxruntime-gpu 2>&1 | tail -n 5", check=False)
if torch.__version__ != torch_before:
    record_failure(
        "environment changed mid-run",
        f"torch {torch_before} -> {torch.__version__} after pip install; "
        "earlier results in this session may not be comparable",
    )

import importlib

for mod in ("ultralytics", "onnx", "onnxruntime"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:12s} {getattr(m, '__version__', '?')}")
    except Exception as exc:  # noqa: BLE001
        print(f"{mod:12s} MISSING ({exc})")
        record_failure(f"import {mod}", repr(exc))

try:
    import onnxruntime as ort
    print("ORT providers:", ort.get_available_providers())
except Exception as exc:  # noqa: BLE001
    print("ORT not importable:", exc)


$ pip -q install ultralytics onnx onnxruntime-gpu 2>&1 | tail -n 5

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics  8.4.218
onnx         1.23.2
onnxruntime  1.32.0
ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [17]:
# ---- 4b. Export YOLOv8n to ONNX, then run the ladder ----
sh(f"python {shlex.quote(str(REPO / 'phase_b' / 'export_model.py'))}", check=False)

rc = sh(
    f"python {shlex.quote(str(REPO / 'phase_b' / 'infer_bench.py'))} "
    f"--imgsz 640 --batch 1 --iters 100 --warmup 20 "
    f"--csv {shlex.quote(str(REPO / 'results' / 'infer.csv'))}",
    check=False,
).returncode
if rc != 0:
    record_failure("inference ladder", f"infer_bench.py exit code {rc}")


$ python /content/cuda-waste-pipeline/phase_b/export_model.py

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from '/content/cuda-waste-pipeline/models/yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

ONNX: starting export with onnx 1.23.2 opset 17...
ONNX: export success ✅ 1.3s, saved as '/content/cuda-waste-pipeline/models/yolov8n.onnx' (12.2 MB)

Export complete (3.8s)
Results saved to /content/cuda-waste-pipeline/models/yolov8n.onnx
Predict:         yolo predict task=detect model=/content/cuda-waste-pipeline/models/yolov8n.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/cuda-waste-pipeline/models/yolov8n.onnx imgsz=640 data=coco.yaml 
Visu

## 4c. TensorRT (stretch, opt-in)

Deliberately its own cell and off by default: `pip install tensorrt` can upgrade
`nvidia-*` packages out from under the preinstalled torch, which would invalidate every
result above. Run this **last**, and if torch breaks afterwards, restart the runtime — the
CSVs are already on disk.


In [18]:
RUN_TENSORRT = False  # flip to True to attempt it

if RUN_TENSORRT:
    rc = sh("pip -q install tensorrt 2>&1 | tail -n 5", check=False).returncode
    if rc != 0:
        record_failure("tensorrt install", "pip install tensorrt failed")
    else:
        rc = sh(
            f"python {shlex.quote(str(REPO / 'phase_b' / 'infer_bench.py'))} "
            f"--only trt_fp16 --iters 100 --warmup 20 "
            f"--csv {shlex.quote(str(REPO / 'results' / 'infer.csv'))}",
            check=False,
        ).returncode
        if rc != 0:
            record_failure("tensorrt backend", "engine build or inference failed")
else:
    print("TensorRT not attempted in this run (RUN_TENSORRT = False).")


TensorRT not attempted in this run (RUN_TENSORRT = False).


## 5. Pipelined loop and batching

Serial vs 2-stream vs 3-stream, then 1/2/4 simulated camera feeds.

The honest limit worth reading: PyTorch and ONNX Runtime run on their own internal streams, so
the pipeline overlaps **host→device copy of frame k+1 with inference of frame k**, plus the
Python-side preprocessing, but it does not claim to run two inferences concurrently on one
session. The measured speedup should be read as "what removing the copy stall bought", and the
write-up says exactly that.


In [19]:
rc = sh(
    f"python {shlex.quote(str(REPO / 'phase_b' / 'pipeline.py'))} "
    f"--backend pt_fp32 --imgsz 640 --frames 300 --warmup 20 "
    f"--csv {shlex.quote(str(REPO / 'results' / 'pipeline.csv'))} "
    f"--batch-csv {shlex.quote(str(REPO / 'results' / 'batching.csv'))}",
    check=False,
).returncode
if rc != 0:
    record_failure("pipelined loop", f"pipeline.py exit code {rc}")


$ python /content/cuda-waste-pipeline/phase_b/pipeline.py --backend pt_fp32 --imgsz 640 --frames 300 --warmup 20 --csv /content/cuda-waste-pipeline/results/pipeline.csv --batch-csv /content/cuda-waste-pipeline/results/batching.csv


KeyboardInterrupt: 

## 6. Profiling

`nsys` gives a timeline and a per-kernel duration table, which is enough to show that the
preprocess kernel is memory-bound and that the GEMM is not. `ncu` would give achieved occupancy
and memory-throughput counters, but it needs `CAP_SYS_ADMIN` for performance counters, which
Colab does not grant — so it is run once and the error is recorded rather than papered over.

Download the `.nsys-rep` files from the Files sidebar and open them in the Nsight Systems GUI
on your own machine; the `.txt` stats are kept in `results/` because the report itself is a
multi-megabyte binary.


In [ ]:
# ---- 6a. Nsight Systems: timeline + kernel table for two representative kernels ----
if shutil.which("nsys"):
    for target, extra in (
        ("vector_add", "--n 20000000 --iters 20"),
        ("matmul", "--n 1024 --cpu-n 64 --iters 5"),
    ):
        cmd = (
            f"nsys profile --force-overwrite true -t cuda,nvtx,osrt "
            f"-o {shlex.quote(str(REPO / 'reports' / target))} "
            f"{shlex.quote(str(REPO / 'build' / target))} {extra} "
            f"--csv {shlex.quote(str(REPO / 'results' / 'profiling_tmp.csv'))}"
        )
        rc = sh(cmd, cwd=REPO / "phase_a", check=False).returncode
        if rc != 0:
            record_failure(f"nsys profile {target}", f"exit code {rc}")
            continue
        out = REPO / "results" / f"profiling_{target}_stats.txt"
        stats = sh(
            f"nsys stats --report cuda_gpu_kern_sum,cuda_gpu_mem_time_sum "
            f"{shlex.quote(str(REPO / 'reports' / (target + '.nsys-rep')))}",
            check=False,
        )
        out.write_text(stats.stdout, encoding="utf-8")
        print(f"wrote {out}")
else:
    record_failure("nsys", "nsys not installed; skipped the timeline")


In [ ]:
# ---- 6b. Nsight Compute: attempted, expected to be refused on Colab ----
if shutil.which("ncu"):
    rc = sh(
        f"ncu --set basic --target-processes all --csv "
        f"--log-file {shlex.quote(str(REPO / 'results' / 'profiling_ncu.csv'))} "
        f"{shlex.quote(str(REPO / 'build' / 'vector_add'))} "
        f"--n 1000000 --iters 5 "
        f"--csv {shlex.quote(str(REPO / 'results' / 'profiling_tmp.csv'))}",
        cwd=REPO / "phase_a",
        check=False,
    )
    if rc.returncode != 0:
        record_failure(
            "ncu (Nsight Compute)",
            "profiling refused - performance counters require privileges Colab does not "
            "grant (ERR_NVGPUCTRPERM). Fell back to CUDA event timings + nsys kernel table.",
        )
else:
    record_failure("ncu (Nsight Compute)", "not installed in this image")


## 7. Summary

Generated by `tools/summary.py` from `results/*.csv`. Missing benchmarks appear as
**NOT RUN**; nothing is interpolated.


In [ ]:
# ---- 7a. Generate and display results/summary.md ----
import importlib.util

spec = importlib.util.spec_from_file_location("summary_tool", REPO / "tools" / "summary.py")
summary_tool = importlib.util.module_from_spec(spec)
spec.loader.exec_module(summary_tool)

summary_tool.main([])
text = (REPO / "results" / "summary.md").read_text(encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(text))
except ImportError:  # not running under IPython
    print(text)


In [ ]:
# ---- 7b. Ship the results back ----
# Two return paths, because the runtime that produced these numbers is disposable:
#   1. results_bundle.zip -> browser download (works in the Colab web UI)
#   2. every CSV printed inline -> saved into this .ipynb file on your own disk
# Path 2 always works, including under the Colab VS Code extension, which has no
# browser download prompt and no access to your local filesystem.
import zipfile

results = REPO / "results"
bundle = results / "results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(results.rglob("*")):
        if path.is_file() and path.suffix != ".zip":
            zf.write(path, path.relative_to(REPO))
print(f"wrote {bundle} ({bundle.stat().st_size / 1024:.1f} KiB)\n")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(bundle))
    except Exception as exc:
        print(f"browser download unavailable ({type(exc).__name__}: {exc})")
        print("use the inline dump below instead\n")

# Always dump the raw CSVs. This output is written into the notebook file itself,
# so the numbers survive even when nothing can be downloaded.
for path in sorted(results.glob("*.csv")):
    print("=" * 70)
    print(f"FILE results/{path.name}")
    print("=" * 70)
    print(path.read_text(encoding="utf-8").rstrip())
    print()

failures = results / "failures.md"
if failures.exists():
    print("=" * 70)
    print("FILE results/failures.md")
    print("=" * 70)
    print(failures.read_text(encoding="utf-8").rstrip())


## What was built, and why

| Milestone | Kernel / script | The one-line point |
|---|---|---|
| 1 | `vector_add.cu` | Memory-bound: 12 bytes moved per add, so the ceiling is bandwidth, not ALU throughput. |
| 2 | `reduction.cu` | Each version removes something: global round trips, then `__syncthreads()`, then bank conflicts. |
| 3 | `matmul.cu` | Tiling turns a memory-bound kernel compute-bound; cuBLAS stays ahead because of register blocking. |
| 4 | `streams.cu` | Pinning is a prerequisite, streams are the mechanism; 4 chunks is too few to hide the tail. |
| 5 | `preprocess.cu` | Five ops in one pass over destination pixels: no intermediate tensor, no five launches. |
| 6 | `infer_bench.py` | FP16 tensor cores on Turing make the precision step the biggest single win. |
| 7 | `pipeline.py` | End-to-end FPS is what a camera loop sees, and it is always below 1/latency of any stage. |
| 8 | `pipeline.py --feeds` | FPS per feed falls once the GPU saturates; batching trades latency for throughput. |

**Limitations, stated plainly:** prototype on a T4 (sm_75, no `cp.async`, no TMA, no BF16);
synthetic or looped footage for the demo; a few epochs of fine-tuning at most, so the detector
is essentially pretrained COCO; Colab's virtualised vCPUs make the CPU baselines noisier than
a bare-metal comparison would be.
